In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  Simuladores del Capítulo 15                                                ║
║  El Control Local del Comportamiento de Elección                            ║
║  Aprendizaje y Comportamiento Adaptable: Principios y Modelos               ║
║  Arturo Bouzas · UNAM                                                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Simulador A  Convergencia ensayo a ensayo (MM vs. Mejoramiento)            ║
║  Simulador B  Secuencias de respuesta y longitudes de carrera               ║
║  Simulador C  Experimento de Williams — programa concurrente RV-IV          ║
╚══════════════════════════════════════════════════════════════════════════════╝

Uso en Colab: Ejecutar las dos celdas en orden.
"""

# ════════════════════════════════════════════════════════════════════════════
# CELDA 1 — Dependencias e instalación
# ════════════════════════════════════════════════════════════════════════════
import sys, subprocess

for pkg in ["ipywidgets", "matplotlib", "numpy"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings("ignore")

print("✓ Dependencias listas — ejecuta la Celda 2 para abrir los simuladores.")


# ════════════════════════════════════════════════════════════════════════════
# CELDA 2 — Motores de simulación, interfaz y visualizaciones
# ════════════════════════════════════════════════════════════════════════════

# ── Paleta y estilo ──────────────────────────────────────────────────────────
AZUL    = "#2C5282"
NARANJA = "#C05621"
VERDE   = "#276749"
GRIS    = "#718096"
GRIS_L  = "#EBF4FF"

def _estilo(ax, title="", ylabel="", xlabel=""):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(GRIS)
    ax.tick_params(colors=GRIS, labelsize=9)
    ax.xaxis.label.set_color(GRIS)
    ax.yaxis.label.set_color(GRIS)
    if title:
        ax.set_title(title, color=AZUL, fontsize=10, pad=6)
    if ylabel:
        ax.set_ylabel(ylabel, color=GRIS, fontsize=9)
    if xlabel:
        ax.set_xlabel(xlabel, color=GRIS, fontsize=9)

def _slider(desc, mn, mx, val, step=1, fmt=None):
    cls = widgets.FloatSlider if fmt else widgets.IntSlider
    return cls(
        description=desc, min=mn, max=mx, value=val, step=step,
        style={"description_width": "220px"},
        layout=widgets.Layout(width="460px"),
        readout_format=fmt or "d",
    )

def _boton(label="▶  Simular"):
    return widgets.Button(
        description=label,
        style={"button_color": AZUL},
        layout=widgets.Layout(width="160px", height="34px"),
    )

def _html_seccion(letra, titulo, descripcion):
    return HTML(f"""
    <div style="background:#2C5282;color:white;padding:14px 18px;border-radius:8px;
                margin:10px 0 6px 0;font-family:Georgia,serif;">
      <span style="font-size:11px;opacity:.75">Simulador {letra}</span>
      <div style="font-size:16px;font-weight:bold;margin-top:2px">{titulo}</div>
    </div>
    <div style="background:#EBF4FF;border-left:4px solid #2C5282;padding:10px 14px;
                border-radius:0 6px 6px 0;font-family:Georgia,serif;font-size:12.5px;
                color:#2D3748;line-height:1.6;margin-bottom:8px">{descripcion}</div>
    """)


# ════════════════════════════════════════════════════════════════════════════
# MOTORES DE SIMULACIÓN
# ════════════════════════════════════════════════════════════════════════════

def _armar_refuerzo(armado, iv, rng):
    """Arma un reforzador con prob 1/IV si no hay uno ya disponible."""
    return armado or (rng.random() < 1.0 / iv)


def simular_mm(iv1, iv2, n_trials, t1_0, semilla):
    """
    Maximización momentánea — regla de Luce sobre probabilidades instantáneas.

    En cada ensayo: P(opción 1) = p1/(p1+p2)
    donde p_i = 1 - (1 - 1/IV_i)^{tiempo desde última visita a i}.

    Devuelve:
        choices       : array de enteros 1/2, longitud n_trials
        T1_running    : T1 acumulada ensayo a ensayo
        p1_arr, p2_arr: probabilidades instantáneas en cada ensayo
    """
    rng = np.random.default_rng(semilla)

    # Inicializar contadores proporcionales a T1_0
    # Si t1_0 bajo → animal ha estado más en opción 2 → t_desde_1 es alto
    t1 = max(2, int((1 - t1_0) * 20))
    t2 = max(2, int(t1_0 * 20))

    armado1 = armado2 = False
    choices  = np.empty(n_trials, dtype=int)
    T1_run   = np.empty(n_trials)
    p1_arr   = np.empty(n_trials)
    p2_arr   = np.empty(n_trials)
    cum1     = 0

    for k in range(n_trials):
        armado1 = _armar_refuerzo(armado1, iv1, rng)
        armado2 = _armar_refuerzo(armado2, iv2, rng)

        p1 = 1.0 - (1.0 - 1.0 / iv1) ** t1
        p2 = 1.0 - (1.0 - 1.0 / iv2) ** t2
        p1_arr[k] = p1
        p2_arr[k] = p2

        den = p1 + p2
        c = 1 if rng.random() < (p1 / den if den > 0 else 0.5) else 2
        choices[k] = c

        if c == 1:
            cum1 += 1; t1 = 1; t2 += 1
            if armado1: armado1 = False
        else:
            t1 += 1; t2 = 1
            if armado2: armado2 = False

        T1_run[k] = cum1 / (k + 1)

    return choices, T1_run, p1_arr, p2_arr


def simular_mel(iv1, iv2, n_trials, t1_0, semilla):
    """
    Mejoramiento — regla de Luce sobre tasas locales de refuerzo.

    En cada ensayo: P(opción 1) = λ1/(λ1+λ2)
    donde λ_i = reforzadores_obtenidos_en_i / tiempo_asignado_a_i.

    Se inicializa con una historia previa proporcional a t1_0 para que
    la trayectoria de convergencia sea visible desde el principio.

    Devuelve:
        choices        : array de enteros 1/2
        T1_running     : T1 acumulada ensayo a ensayo
        local1, local2 : tasas locales en cada ensayo
    """
    rng = np.random.default_rng(semilla)

    # Historia previa: da un punto de partida sesgado
    init = 30
    T1 = t1_0 * init + 0.5          # tiempo acumulado en opción 1
    T2 = (1 - t1_0) * init + 0.5   # tiempo acumulado en opción 2
    r1 = T1 / iv1                   # reforzadores esperados
    r2 = T2 / iv2

    armado1 = armado2 = False
    choices  = np.empty(n_trials, dtype=int)
    T1_run   = np.empty(n_trials)
    loc1_arr = np.empty(n_trials)
    loc2_arr = np.empty(n_trials)
    cum1     = t1_0 * init
    cum_tot  = float(init)

    for k in range(n_trials):
        armado1 = _armar_refuerzo(armado1, iv1, rng)
        armado2 = _armar_refuerzo(armado2, iv2, rng)

        lam1 = r1 / T1
        lam2 = r2 / T2
        loc1_arr[k] = lam1
        loc2_arr[k] = lam2

        den = lam1 + lam2
        c = 1 if rng.random() < (lam1 / den if den > 0 else 0.5) else 2
        choices[k] = c

        if c == 1:
            T1 += 1; cum1 += 1
            if armado1: r1 += 1; armado1 = False
        else:
            T2 += 1
            if armado2: r2 += 1; armado2 = False

        cum_tot += 1
        T1_run[k] = cum1 / cum_tot

    return choices, T1_run, loc1_arr, loc2_arr


def p_cambio_por_carrera(choices, max_run=8):
    """
    Para cada longitud de carrera n, calcula P(cambiar en el ensayo n de la carrera).
    Esto es la tasa de hazard empírica del cambio: dada una carrera de longitud
    al menos n, ¿con qué probabilidad termina exactamente en n?
    """
    # Registrar: (posición_en_carrera, ¿cambió_en_siguiente?)
    pares = []
    pos   = 1
    for i in range(len(choices) - 1):
        cambio = choices[i] != choices[i + 1]
        pares.append((pos, int(cambio)))
        pos = 1 if cambio else pos + 1

    pares = np.array(pares)
    ns, ps = [], []
    for n in range(1, max_run + 1):
        mask = pares[:, 0] == n
        if mask.sum() >= 8:
            ns.append(n)
            ps.append(pares[mask, 1].mean())
    return np.array(ns), np.array(ps)


def p_cambio_mm_teorica(iv1, iv2, max_run=8):
    """
    Predicción teórica del modelo MM: P(cambiar en ensayo n de la carrera).
    Suponemos que el organismo lleva n ensayos en la misma opción (digamos op. 1)
    y t_desde_2 = n. La probabilidad de elegir op. 2 es p2/(p1+p2) con:
        p1 = 1/iv1 (recién visitada, t_desde_1=1)
        p2 = 1 - (1-1/iv2)^n
    Promedio sobre las dos opciones.
    """
    ns = np.arange(1, max_run + 1)
    # Si se está en opción 1 (t_desde_2 = n):
    p1_a = 1.0 / iv1
    p2_a = 1.0 - (1.0 - 1.0 / iv2) ** ns
    prob_cambio_a = p2_a / (p1_a + p2_a)

    # Si se está en opción 2 (t_desde_1 = n):
    p2_b = 1.0 / iv2
    p1_b = 1.0 - (1.0 - 1.0 / iv1) ** ns
    prob_cambio_b = p1_b / (p2_b + p1_b)

    return ns, (prob_cambio_a + prob_cambio_b) / 2


def simular_williams(p_rv, media_vi, n_trials, t1_0, semilla):
    """
    Programa concurrente RV-IV de ensayos discretos (Williams, 1985).

    RV : refuerzo con probabilidad constante p_rv en cada ensayo.
    IV : probabilidad de refuerzo crece con el tiempo desde la última
         respuesta al IV (proceso geométrico acumulado).

    Simula dos agentes en paralelo:
      · MM  : elige proporcionalmente a las probabilidades momentáneas
      · Obs : elige con sesgo por perseverancia (ignora t_desde_vi),
              reproduce el patrón observado empíricamente por Williams.

    Devuelve, por bin de t_desde_vi (1..max_t):
        t_bins         : valores del eje x
        p_ref_disponible: P(refuerzo IV disponible | t_desde_vi) — teórica
        p_vi_mm        : P(respuesta al IV | t_desde_vi) bajo MM
        p_vi_obs       : P(respuesta al IV | t_desde_vi) observado
    """
    rng = np.random.default_rng(semilla)
    max_t = 10

    # Estado inicial proporcional a t1_0 (usa más RV al inicio si t1_0 > 0.5)
    t_desde_vi = max(1, int((1 - t1_0) * max_t))

    # Acumuladores: para cada t_desde_vi, contamos respuestas y vi_choices
    n_obs        = np.zeros(max_t + 1, dtype=int)
    vi_mm_count  = np.zeros(max_t + 1, dtype=int)
    vi_obs_count = np.zeros(max_t + 1, dtype=int)
    vi_armado    = False

    # Parámetro de perseverancia para el agente "observado"
    # El agente observado mezcla: con prob eta sigue perseverando,
    # con prob (1-eta) elige aleatoriamente ignorando t_desde_vi.
    eta_persist = 0.55
    ultima_eleccion = "RV"  # última respuesta del agente observado

    for _ in range(n_trials):
        vi_armado = _armar_refuerzo(vi_armado, media_vi, rng)

        t_bin = min(t_desde_vi, max_t)
        n_obs[t_bin] += 1

        # Probabilidad momentánea de refuerzo en VI dado t_desde_vi
        p_vi_inst = 1.0 - (1.0 - 1.0 / media_vi) ** t_desde_vi

        # ── Agente MM ──────────────────────────────────────────────────────
        den_mm = p_rv + p_vi_inst
        prob_vi_mm = p_vi_inst / den_mm if den_mm > 0 else 0.5
        elige_vi_mm = rng.random() < prob_vi_mm
        vi_mm_count[t_bin] += int(elige_vi_mm)

        # ── Agente observado (perseverancia) ───────────────────────────────
        # Mezcla: con prob eta_persist repite la última elección,
        # de lo contrario elige según tasas globales (ignora t_desde_vi).
        prob_global_vi = media_vi / (media_vi + 1.0 / p_rv)
        if rng.random() < eta_persist:
            elige_vi_obs = ultima_eleccion == "VI"
        else:
            elige_vi_obs = rng.random() < prob_global_vi
        vi_obs_count[t_bin] += int(elige_vi_obs)
        ultima_eleccion = "VI" if elige_vi_obs else "RV"

        # ── Actualizar estado (basado en agente MM para t_desde_vi) ────────
        if elige_vi_mm:
            if vi_armado: vi_armado = False
            t_desde_vi = 1
        else:
            t_desde_vi += 1

    # ── Computar proporciones por bin ─────────────────────────────────────
    t_bins = np.arange(1, max_t + 1)
    p_ref  = 1.0 - (1.0 - 1.0 / media_vi) ** t_bins  # teórica
    p_vi_mm_out  = np.where(n_obs[1:] > 5,
                            vi_mm_count[1:]  / np.maximum(n_obs[1:], 1), np.nan)
    p_vi_obs_out = np.where(n_obs[1:] > 5,
                            vi_obs_count[1:] / np.maximum(n_obs[1:], 1), np.nan)

    return t_bins, p_ref, p_vi_mm_out, p_vi_obs_out


# ════════════════════════════════════════════════════════════════════════════
# SIMULADOR A — Convergencia ensayo a ensayo
# ════════════════════════════════════════════════════════════════════════════

display(_html_seccion("A",
    "Convergencia ensayo a ensayo",
    "Compara la trayectoria de T₁ respuesta por respuesta bajo los dos modelos. "
    "El <b style='color:#276749'>mejoramiento</b> parte del punto inicial que configures y "
    "converge gradualmente hacia igualación porque las tasas locales "
    "se van igualando a medida que se acumula experiencia. "
    "La <b style='color:#C05621'>maximización momentánea</b> es sensible desde el "
    "primer ensayo a las probabilidades instantáneas, por lo que fluctúa "
    "alrededor del punto de igualación casi inmediatamente. "
    "El panel derecho muestra las <i>tasas locales</i> del mejoramiento "
    "y las <i>probabilidades instantáneas</i> de la MM convergiendo a lo largo de la sesión."
))

iv1_A = _slider("IV₁ (media, en ensayos)",   5, 60, 15, 5)
iv2_A = _slider("IV₂ (media, en ensayos)",   5, 60, 30, 5)
t1_A  = _slider("Proporción inicial T₁",     0.05, 0.95, 0.15, 0.05, ".2f")
n_A   = _slider("Ensayos por sesión",        100, 800, 300, 50)
sem_A = _slider("Semilla aleatoria",          1, 200, 42)
mod_A = widgets.RadioButtons(
    options=["Ambos modelos", "Solo mejoramiento", "Solo maximización momentánea"],
    description="", layout=widgets.Layout(margin="4px 0"),
)
btn_A = _boton()
out_A = widgets.Output()

display(widgets.VBox([
    widgets.HBox([iv1_A, iv2_A]),
    widgets.HBox([t1_A, n_A]),
    widgets.HBox([sem_A, mod_A]),
    btn_A, out_A,
]))


def graficar_A(_):
    with out_A:
        clear_output(wait=True)
        iv1, iv2 = iv1_A.value, iv2_A.value
        t1_0  = t1_A.value
        N     = n_A.value
        sem   = sem_A.value
        T1_ig = iv2 / (iv1 + iv2)
        mod   = mod_A.value

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
        fig.subplots_adjust(wspace=0.35)

        trials = np.arange(1, N + 1)

        # ── Panel izquierdo: T1 vs ensayo ─────────────────────────────────
        if mod != "Solo maximización momentánea":
            _, T1_mel, loc1, loc2 = simular_mel(iv1, iv2, N, t1_0, sem)
            ax1.plot(trials, T1_mel, color=VERDE, lw=1.8,
                     label="Mejoramiento", alpha=0.9)
            # Suavizar locales para panel derecho
            win = max(1, N // 40)
            kernel = np.ones(win) / win
            ax2.plot(trials[:len(np.convolve(loc1, kernel, "valid"))],
                     np.convolve(loc1, kernel, "valid"),
                     color=AZUL, lw=1.5, label="Tasa local op. 1 (mel.)")
            ax2.plot(trials[:len(np.convolve(loc2, kernel, "valid"))],
                     np.convolve(loc2, kernel, "valid"),
                     color=VERDE, lw=1.5, ls="--", label="Tasa local op. 2 (mel.)")

        if mod != "Solo mejoramiento":
            _, T1_mm, p1a, p2a = simular_mm(iv1, iv2, N, t1_0, sem + 1)
            ax1.plot(trials, T1_mm, color=NARANJA, lw=1.8,
                     label="Maximización momentánea", alpha=0.9)
            win = max(1, N // 40)
            kernel = np.ones(win) / win
            ax2.plot(trials[:len(np.convolve(p1a, kernel, "valid"))],
                     np.convolve(p1a, kernel, "valid"),
                     color=NARANJA, lw=1.5, label="P momentánea op. 1 (MM)")
            ax2.plot(trials[:len(np.convolve(p2a, kernel, "valid"))],
                     np.convolve(p2a, kernel, "valid"),
                     color="#E53E3E", lw=1.5, ls="--",
                     label="P momentánea op. 2 (MM)")

        ax1.axhline(T1_ig, color=GRIS, lw=1.4, ls=":",
                    label=f"Igualación predicha (T₁* = {T1_ig:.2f})")
        ax1.axhline(t1_0,  color=GRIS, lw=1, ls="--", alpha=0.45,
                    label=f"T₁ inicial ({t1_0:.2f})")
        ax1.set_xlim(0, N); ax1.set_ylim(0, 1)
        ax1.set_yticks([0, 0.2, 0.4, T1_ig, 0.8, 1.0])
        ax1.set_yticklabels(["0", "0.2", "0.4", f"{T1_ig:.2f}", "0.8", "1.0"])
        ax1.legend(frameon=False, fontsize=8.5)
        _estilo(ax1, "Proporción de respuestas a opción 1 (T₁)",
                "T₁ acumulada", "Ensayo")

        ax2.set_xlim(0, N)
        ax2.legend(frameon=False, fontsize=8.5)
        _estilo(ax2, "Tasas locales / probabilidades momentáneas",
                "Valor (ref/ensayo  ó  probabilidad)", "Ensayo")

        fig.suptitle(f"IV₁ = {iv1}  ·  IV₂ = {iv2}  ·  T₁ inicial = {t1_0:.2f}",
                     color=GRIS, fontsize=9, y=1.01)
        plt.tight_layout()
        plt.show()

btn_A.on_click(graficar_A)
graficar_A(None)


# ════════════════════════════════════════════════════════════════════════════
# SIMULADOR B — Secuencias de respuesta y longitudes de carrera
# ════════════════════════════════════════════════════════════════════════════

display(_html_seccion("B",
    "Secuencias de respuesta y longitudes de carrera",
    "Muestra la estructura ensayo-a-ensayo de las elecciones y la predicción "
    "central del modelo de maximización momentánea: la <i>probabilidad de cambiar "
    "de opción debe aumentar con el número de respuestas consecutivas</i> a la "
    "opción actual, porque la probabilidad de refuerzo en la alternativa crece "
    "con cada ensayo adicional. "
    "El panel inferior compara esa predicción teórica con lo que produce "
    "cada modelo en la simulación. El patrón observado por Nevin y Silberberg "
    "es una función <b>plana o decreciente</b> — opuesta a la predicción."
))

iv1_B = _slider("IV₁ (media, en ensayos)",   5, 60, 15, 5)
iv2_B = _slider("IV₂ (media, en ensayos)",   5, 60, 30, 5)
t1_B  = _slider("Proporción inicial T₁",     0.05, 0.95, 0.50, 0.05, ".2f")
n_B   = _slider("Ensayos a simular",         200, 1000, 500, 100)
vis_B = _slider("Ensayos a visualizar (raster)", 40, 120, 80, 10)
sem_B = _slider("Semilla aleatoria",           1, 200, 42)
btn_B = _boton()
out_B = widgets.Output()

display(widgets.VBox([
    widgets.HBox([iv1_B, iv2_B]),
    widgets.HBox([t1_B, n_B]),
    widgets.HBox([vis_B, sem_B]),
    btn_B, out_B,
]))


def graficar_B(_):
    with out_B:
        clear_output(wait=True)
        iv1, iv2 = iv1_B.value, iv2_B.value
        t1_0  = t1_B.value
        N     = n_B.value
        N_vis = vis_B.value
        sem   = sem_B.value
        T1_ig = iv2 / (iv1 + iv2)

        # Simular ambos modelos
        ch_mm,  _, _, _  = simular_mm(iv1, iv2, N, t1_0, sem)
        ch_mel, _, _, _  = simular_mel(iv1, iv2, N, t1_0, sem + 1)

        # Longitudes de carrera y P(cambiar | longitud n)
        ns_mm,  ps_mm  = p_cambio_por_carrera(ch_mm)
        ns_mel, ps_mel = p_cambio_por_carrera(ch_mel)
        ns_teo, ps_teo = p_cambio_mm_teorica(iv1, iv2)

        fig = plt.figure(figsize=(12, 6.5))
        gs  = gridspec.GridSpec(2, 3, figure=fig,
                                hspace=0.50, wspace=0.38,
                                height_ratios=[1, 1.2])

        ax_rast_mm  = fig.add_subplot(gs[0, 0])
        ax_rast_mel = fig.add_subplot(gs[0, 1])
        ax_dist_mm  = fig.add_subplot(gs[0, 2])
        ax_hazard   = fig.add_subplot(gs[1, :])

        # ── Rasters ───────────────────────────────────────────────────────
        tail_mm  = ch_mm[-N_vis:]
        tail_mel = ch_mel[-N_vis:]
        xs = np.arange(N_vis)

        for ax, seq, title in [
            (ax_rast_mm,  tail_mm,  "Maximización momentánea"),
            (ax_rast_mel, tail_mel, "Mejoramiento"),
        ]:
            for i, c in enumerate(seq):
                col = AZUL if c == 1 else NARANJA
                ax.barh(0, 1, left=i, height=0.6,
                        color=col, linewidth=0, alpha=0.85)
            ax.set_xlim(0, N_vis)
            ax.set_ylim(-0.5, 0.5)
            ax.set_yticks([]); ax.set_xlabel("Ensayo", color=GRIS, fontsize=8.5)
            ax.tick_params(labelsize=8)
            _estilo(ax, title)
            # Leyenda de colores
            ax.plot([], [], "s", color=AZUL,   label=f"Opción 1 (IV {iv1})")
            ax.plot([], [], "s", color=NARANJA, label=f"Opción 2 (IV {iv2})")
            ax.legend(loc="upper right", frameon=False, fontsize=7.5,
                      markerscale=1.2)

        # ── Distribución de longitudes de carrera (MM) ────────────────────
        runs_mm = []
        run = 1
        for i in range(1, len(ch_mm)):
            if ch_mm[i] == ch_mm[i-1]: run += 1
            else: runs_mm.append(run); run = 1
        runs_mm = np.array(runs_mm)
        max_show = min(12, runs_mm.max() if len(runs_mm) > 0 else 12)
        bins = np.arange(0.5, max_show + 1.5)
        ax_dist_mm.hist(np.clip(runs_mm, 1, max_show), bins=bins,
                        color=NARANJA, alpha=0.75, edgecolor="white", lw=0.5)
        ax_dist_mm.set_xlabel("Longitud de carrera", color=GRIS, fontsize=8.5)
        _estilo(ax_dist_mm, "Distribución de carreras (MM)", "Frecuencia")

        # ── Panel principal: P(cambiar | longitud n) ──────────────────────
        w = 0.25
        ax_hazard.bar(ns_teo - w,  ps_teo, width=w, color=GRIS,  alpha=0.70,
                      label="Predicción teórica MM (creciente)")
        ax_hazard.bar(ns_mm,       ps_mm,  width=w, color=NARANJA, alpha=0.80,
                      label="MM simulada (Luce sobre p instantáneas)")
        ax_hazard.bar(ns_mel + w,  ps_mel, width=w, color=VERDE, alpha=0.80,
                      label="Mejoramiento simulado")

        # Línea de tendencia observada (Nevin): plana/decreciente
        ax_hazard.plot(np.arange(1, 9),
                       0.28 - 0.018 * np.arange(1, 9),
                       "o--", color=AZUL, lw=1.8, ms=5,
                       label="Esquema del patrón observado (Nevin, 1969) — plano/decreciente")

        ax_hazard.set_xticks(np.arange(1, max(ns_teo.max(), 8) + 1))
        ax_hazard.set_ylim(0, 1); ax_hazard.set_xlim(0.4, 8.8)
        ax_hazard.legend(frameon=False, fontsize=8.5, ncol=2)
        _estilo(ax_hazard,
                "P(cambiar de opción | n respuestas consecutivas a la opción actual)",
                "P (cambio)", "Longitud de carrera n")

        # Anotación clave
        ax_hazard.annotate(
            "↑ El modelo predice\nque debería crecer",
            xy=(5, ps_teo[4]), xytext=(5.3, ps_teo[4] + 0.12),
            fontsize=8, color=GRIS,
            arrowprops=dict(arrowstyle="->", color=GRIS, lw=1),
        )
        ax_hazard.annotate(
            "↓ Lo observado:\nplana o decreciente",
            xy=(4, 0.28 - 0.018 * 4), xytext=(4.3, 0.05),
            fontsize=8, color=AZUL,
            arrowprops=dict(arrowstyle="->", color=AZUL, lw=1),
        )

        plt.suptitle(f"IV₁ = {iv1}  ·  IV₂ = {iv2}  ·  T₁* = {T1_ig:.2f}",
                     color=GRIS, fontsize=9)
        plt.show()

btn_B.on_click(graficar_B)
graficar_B(None)


# ════════════════════════════════════════════════════════════════════════════
# SIMULADOR C — Experimento de Williams: programa concurrente RV-IV
# ════════════════════════════════════════════════════════════════════════════

display(_html_seccion("C",
    "Experimento de Williams — Concurrente RV-IV",
    "Reproduce la lógica del experimento de Williams (1985). "
    "El programa RV tiene probabilidad de refuerzo <i>constante</i> en cada ensayo; "
    "el programa IV tiene probabilidad de refuerzo que <i>crece</i> con el número de "
    "ensayos desde la última respuesta al IV. "
    "El modelo de maximización momentánea predice que la probabilidad de "
    "responder al IV debe aumentar con ese contador (Panel B). "
    "El patrón observado es una función <b>plana</b> — el organismo no sigue "
    "la probabilidad momentánea del IV aunque esta sea creciente (Panel A)."
))

p_rv_C   = _slider("P(refuerzo) en RV",           0.05, 0.40, 0.15, 0.05, ".2f")
med_vi_C = _slider("Media del IV (en ensayos)",      5,   40,   15,    5)
t1_C     = _slider("Fracción inicial en RV",        0.05, 0.95, 0.70, 0.05, ".2f")
n_C      = _slider("Ensayos totales",              500, 5000, 2000,  500)
sem_C    = _slider("Semilla aleatoria",              1,  200,   42)
btn_C    = _boton()
out_C    = widgets.Output()

display(widgets.VBox([
    widgets.HBox([p_rv_C, med_vi_C]),
    widgets.HBox([t1_C, n_C]),
    sem_C, btn_C, out_C,
]))


def graficar_C(_):
    with out_C:
        clear_output(wait=True)
        p_rv   = p_rv_C.value
        med_vi = med_vi_C.value
        N      = n_C.value
        t1_0   = t1_C.value
        sem    = sem_C.value

        t_bins, p_ref, p_vi_mm, p_vi_obs = simular_williams(
            p_rv, med_vi, N, t1_0, sem)

        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4.2))
        fig.subplots_adjust(wspace=0.38)

        # ── Panel A: P(refuerzo disponible en IV) — teórica + simulada ───
        ax1.plot(t_bins, p_ref, "-", color=AZUL, lw=2.0,
                 label="Teórica: P = 1−(1−1/IV)ⁿ")
        ax1.axhline(p_rv, color=NARANJA, lw=1.8, ls="--",
                    label=f"P(refuerzo RV) = {p_rv:.2f} (constante)")
        ax1.fill_between(t_bins, p_rv, p_ref,
                         where=p_ref > p_rv, alpha=0.12, color=AZUL,
                         label="IV más probable que RV")
        ax1.set_xlim(0.5, t_bins[-1] + 0.5)
        ax1.set_ylim(0, 1)
        ax1.set_xticks(t_bins)
        ax1.legend(frameon=False, fontsize=8.5)
        _estilo(ax1,
                "Panel A — Probabilidad de refuerzo disponible en IV",
                "P (refuerzo disponible)",
                "Ensayos desde última respuesta al IV")

        # ── Panel B: P(respuesta al IV) — modelo vs. observado ────────────
        valid = ~np.isnan(p_vi_mm) & ~np.isnan(p_vi_obs)
        w = 0.3
        ax2.bar(t_bins[valid] - w/2, p_vi_mm[valid],  width=w,
                color=NARANJA, alpha=0.80,
                label="Maximización momentánea (predice ↑)")
        ax2.bar(t_bins[valid] + w/2, p_vi_obs[valid], width=w,
                color=VERDE,   alpha=0.80,
                label="Patrón observado (plano)")
        ax2.plot(t_bins[valid],
                 np.full(valid.sum(), np.nanmean(p_vi_obs[valid])),
                 ":", color=GRIS, lw=1.4)
        ax2.set_xlim(0.5, t_bins[-1] + 0.5)
        ax2.set_ylim(0, 1)
        ax2.set_xticks(t_bins)
        ax2.legend(frameon=False, fontsize=8.5)
        _estilo(ax2,
                "Panel B — Probabilidad de responder al IV",
                "P (respuesta al IV)",
                "Ensayos desde última respuesta al IV")

        # ── Panel C: igualación global ─────────────────────────────────────
        # Tasa de refuerzo esperada por opción
        r_rv = p_rv
        r_vi = 1.0 / med_vi
        T1_ig = r_rv / (r_rv + r_vi)   # fracción en RV en igualación
        prop_vi_mm  = 1 - np.nanmean(p_vi_mm[valid])   # fracción en RV
        prop_vi_obs = 1 - np.nanmean(p_vi_obs[valid])

        categorias = ["Igualación\npredicha", "MM\n(simulada)", "Observado\n(simulado)"]
        valores    = [T1_ig, prop_vi_mm, prop_vi_obs]
        colores    = [GRIS, NARANJA, VERDE]
        bars = ax3.bar(categorias, valores, color=colores, alpha=0.80, width=0.55)
        for bar, val in zip(bars, valores):
            ax3.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                     f"{val:.2f}", ha="center", va="bottom",
                     fontsize=9, color=GRIS)
        ax3.set_ylim(0, 1)
        ax3.axhline(T1_ig, color=GRIS, lw=1.2, ls=":")
        _estilo(ax3,
                "Panel C — Fracción de tiempo en RV (global)",
                "Fracción de respuestas en RV", "")

        fig.suptitle(
            f"Concurrente RV(p={p_rv})  —  IV(media={med_vi} ensayos)  "
            f"·  Sesión de {N} ensayos",
            color=GRIS, fontsize=9)
        plt.tight_layout()
        plt.show()

btn_C.on_click(graficar_C)
graficar_C(None)


# ════════════════════════════════════════════════════════════════════════════
# EJERCICIOS
# ════════════════════════════════════════════════════════════════════════════

display(HTML("""
<div style="background:#F7FAFC;border:1px solid #CBD5E0;border-radius:8px;
            padding:18px 22px;font-family:Georgia,serif;font-size:13px;
            color:#2D3748;line-height:1.7;margin-top:20px;">
  <div style="font-size:15px;font-weight:bold;color:#2C5282;margin-bottom:14px;">
    Guía de ejercicios
  </div>

  <b>Simulador A — Ejercicio 1 (básico):</b> Igualación desde puntos distintos<br>
  Configura IV₁=15, IV₂=30. Corre con T₁ inicial = 0.10 y luego con T₁ inicial = 0.90.
  ¿Ambos modelos convergen al mismo T₁*? ¿Cuál tarda más? ¿Por qué el mejoramiento
  muestra una curva en S mientras la MM fluctúa desde el primer ensayo?
  <br><br>

  <b>Simulador A — Ejercicio 2 (intermedio):</b> Programas iguales<br>
  Cambia a IV₁=20, IV₂=20. Predice antes de simular: ¿a qué T₁* deberían
  converger? ¿Qué diferencia verías entre los dos modelos con programas simétricos?
  <br><br>

  <b>Simulador B — Ejercicio 3 (intermedio):</b> La predicción vs. lo observado<br>
  Con IV₁=15, IV₂=45, observa el panel inferior de P(cambiar | longitud n).
  Identifica: (a) cuál de las tres barras es la predicción teórica del modelo MM;
  (b) cuál reproduce el patrón de Nevin; (c) qué forma tiene la curva del mejoramiento.
  ¿Por qué el mejoramiento produce un patrón diferente al de la MM?
  <br><br>

  <b>Simulador B — Ejercicio 4 (avanzado):</b> Diseño experimental<br>
  El panel del raster muestra las secuencias de elección de los dos modelos.
  Compáralas visualmente. Si tuvieras datos de un experimento real y quisieras
  distinguir entre los dos modelos, ¿qué medida calcularías? ¿Cuántos ensayos
  necesitarías para tener suficiente poder estadístico para detectar la diferencia?
  <br><br>

  <b>Simulador C — Ejercicio 5 (intermedio):</b> La disociación de Williams<br>
  Configura p_RV=0.15, media IV=15. Observa los Paneles A y B juntos.
  ¿A partir de cuántos ensayos desde la última respuesta IV la probabilidad de
  refuerzo en el IV supera a la del RV (Panel A)? ¿En ese punto, qué hace el
  agente MM? ¿Y el agente observado? ¿Qué conclusión extraes sobre si los
  organismos reales usan maximización momentánea?
  <br><br>

  <b>Simulador C — Ejercicio 6 (avanzado):</b> Igualación sin maximización momentánea<br>
  Varía la media del IV entre 10 y 40 manteniendo p_RV=0.15. Observa el
  Panel C: ¿el agente "observado" (con perseverancia) igualiza globalmente
  aunque no maximice momento a momento? ¿Qué implica esto sobre la relación
  entre igualación global y el mecanismo local que la genera?
</div>
"""))

✓ Dependencias listas — ejecuta la Celda 2 para abrir los simuladores.
